# Прогноз количества запросов на следующий час в отдельном регионе

Подготовка почасового ряда, признаков

Модели: `baseline`, `linear`, `rf`, `cat`


In [2]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["axes.grid"] = True


## Настройки


In [3]:
DATA_PATH = "df_with_cat.csv"
city_ids = [34, 16, 35, 6, 7, 42, 1, 38, 21, 66, 41, 27]
CITY_ID = city_ids[0]

TEST_SIZE = 0.20
VALID_SIZE = 0.20
RANDOM_STATE = 42

RF_TREES = 500
CAT_ITERATIONS = 3000


## Подготовка данных


In [4]:
events = pd.read_csv(DATA_PATH)

In [5]:
events["model_response_timestamp"] = pd.to_datetime(
    events["model_response_timestamp"],
    unit="s",
)
events["date_hour"] = events["model_response_timestamp"].dt.floor("h")

hourly = (
    events.groupby(["number", "date_hour"])
    .size()
    .reset_index(name="count")
)

city_hourly = hourly.loc[hourly["number"] == CITY_ID, ["date_hour", "count"]]
all_hours = pd.date_range(
    city_hourly["date_hour"].min(),
    city_hourly["date_hour"].max(),
    freq="h",
)

base_df = (
    city_hourly.set_index("date_hour")
    .reindex(all_hours, fill_value=0)
    .rename_axis("date_hour")
    .reset_index()
)

base_df = base_df.sort_values("date_hour").reset_index(drop=True)

base_df.head()


,date_hour,count
0,2026-02-13 21:00:00,86
1,2026-02-13 22:00:00,52
2,2026-02-13 23:00:00,37
3,2026-02-14 00:00:00,12
4,2026-02-14 01:00:00,6


In [6]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values("date_hour").reset_index(drop=True).copy()

    result["hour"] = result["date_hour"].dt.hour
    result["day"] = result["date_hour"].dt.day
    result["month"] = result["date_hour"].dt.month
    result["day_of_week"] = result["date_hour"].dt.dayofweek
    result["is_weekend"] = result["day_of_week"].isin([5, 6]).astype(int)

    result["hour_sin"] = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"] = np.cos(2 * np.pi * result["hour"] / 24)

    for day in range(7):
        result[f"day_of_week_{day}"] = (result["day_of_week"] == day).astype(int)

    return result


def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values("date_hour").reset_index(drop=True).copy()

    result["lag_1"] = result["count"].shift(1)
    result["lag_2"] = result["count"].shift(2)
    result["lag_24"] = result["count"].shift(24)

    result["rolling_mean_3"] = result["count"].shift(1).rolling(3).mean()
    result["rolling_mean_6"] = result["count"].shift(1).rolling(6).mean()
    result["rolling_mean_24"] = result["count"].shift(1).rolling(24).mean()

    return result


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    return add_lag_features(add_time_features(df))


In [7]:
featured_df = build_features(base_df)
featured_df["predict_1h"] = featured_df["count"]

excluded_dates = featured_df["month"].eq(2) & featured_df["day"].isin([25])
featured_df = featured_df.loc[~excluded_dates].reset_index(drop=True)

day_columns = [f"day_of_week_{day}" for day in range(7)]
feature_cols = [
    "hour_sin",
    "hour_cos",
    "lag_1",
    "lag_2",
    "lag_24",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_mean_24",
    "is_weekend",
    *day_columns,
]

model_df = featured_df.dropna(subset=feature_cols + ["predict_1h"]).reset_index(drop=True)
model_df[feature_cols].head()


,hour_sin,hour_cos,lag_1,lag_2,lag_24,rolling_mean_3,rolling_mean_6,rolling_mean_24,is_weekend,day_of_week_0,day_of_week_1,day_of_week_2,day_of_week_3,day_of_week_4,day_of_week_5,day_of_week_6
0,-0.707107,0.707107,62.0,54.0,86.0,57.666667,54.833333,50.666667,1,0,0,0,0,0,1,0
1,-0.500000,0.866025,63.0,62.0,52.0,59.666667,53.666667,49.708333,1,0,0,0,0,0,1,0
2,-0.258819,0.965926,30.0,63.0,37.0,51.666667,51.833333,48.791667,1,0,0,0,0,0,1,0
3,0.000000,1.000000,24.0,30.0,12.0,39.000000,48.333333,48.250000,1,0,0,0,0,0,0,1
4,0.258819,0.965926,15.0,24.0,6.0,23.000000,41.333333,48.375000,1,0,0,0,0,0,0,1


## Разделение на train / validation / test


In [8]:
split_index = int(len(model_df) * (1 - TEST_SIZE))
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

valid_index = int(len(train_df) * (1 - VALID_SIZE))
cat_train_df = train_df.iloc[:valid_index].copy()
cat_valid_df = train_df.iloc[valid_index:].copy()

X_train = train_df[feature_cols]
y_train = train_df["predict_1h"]

X_cat_train = cat_train_df[feature_cols]
y_cat_train = cat_train_df["predict_1h"]
X_cat_valid = cat_valid_df[feature_cols]
y_cat_valid = cat_valid_df["predict_1h"]

X_test = test_df[feature_cols]
y_test = test_df["predict_1h"]

print(f"train: {X_train.shape}")
print(f"cat validation: {X_cat_valid.shape}")
print(f"test: {X_test.shape}")


train: (576, 16)
cat validation: (116, 16)
test: (144, 16)


## Метрики


In [9]:
scores = []
preds_df = pd.DataFrame({
    "date_hour": test_df["date_hour"].values,
    "true": y_test.values,
})


def save_prediction(name: str, y_true, y_pred) -> None:
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    preds_df[name] = y_pred

    scores.append({
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    })


def print_score(name: str, y_true, y_pred) -> None:
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)

    print(f'''
        "model": {name},
        "MAE": {mean_absolute_error(y_true, y_pred)},
        "RMSE": {np.sqrt(mean_squared_error(y_true, y_pred))},
        "R2": {r2_score(y_true, y_pred)},
        "MAPE": {mean_absolute_percentage_error(y_true, y_pred)},
    ''')

## Baseline

Наивный прогноз: считаем, что в следующий час будет столько же запросов, сколько в текущий.


In [10]:
y_pred_baseline = X_test["lag_1"].values
save_prediction("baseline", y_test, y_pred_baseline)
print_score("baseline", y_test, y_pred_baseline)



        "model": baseline,
        "MAE": 6.5,
        "RMSE": 8.427797919847022,
        "R2": 0.30364627436629543,
        "MAPE": 0.549318048897032,
    


## Linear Regression


In [11]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)
save_prediction("linear", y_test, y_pred_linear)
print_score("linear", y_test, y_pred_linear)



        "model": linear,
        "MAE": 5.177997664392206,
        "RMSE": 6.503359258190408,
        "R2": 0.5853542979525145,
        "MAPE": 0.5497476092703475,
    


## Random Forest


In [12]:
rf_model = RandomForestRegressor(
    n_estimators=RF_TREES,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
save_prediction("rf", y_test, y_pred_rf)
print_score("rf", y_test, y_pred_rf)



        "model": rf,
        "MAE": 5.116079231774034,
        "RMSE": 6.340481638172892,
        "R2": 0.6058639365685554,
        "MAPE": 0.5467968360966238,
    


## CatBoost


In [13]:
cat_model = CatBoostRegressor(
    iterations=CAT_ITERATIONS,
    learning_rate=0.01,
    depth=8,
    l2_leaf_reg=3,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=RANDOM_STATE,
    early_stopping_rounds=100,
    verbose=200,
)
cat_model.fit(
    X_cat_train,
    y_cat_train,
    eval_set=(X_cat_valid, y_cat_valid),
    use_best_model=True,
)

y_pred_cat = cat_model.predict(X_test)
save_prediction("cat", y_test, y_pred_cat)


0:	learn: 12.9311132	test: 12.6501990	best: 12.6501990 (0)	total: 164ms	remaining: 8m 10s
200:	learn: 7.0524882	test: 8.4347201	best: 8.4347201 (200)	total: 1.01s	remaining: 14.1s
400:	learn: 5.8472505	test: 8.0115027	best: 8.0115027 (400)	total: 1.81s	remaining: 11.7s
600:	learn: 5.1235993	test: 7.9310616	best: 7.9297306 (558)	total: 2.58s	remaining: 10.3s
800:	learn: 4.6040178	test: 7.9294942	best: 7.9225973 (725)	total: 3.35s	remaining: 9.2s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 7.922597282
bestIteration = 725

Shrink model to first 726 iterations.


## Сводная таблица


In [14]:
scores_df = pd.DataFrame(scores).sort_values("RMSE").reset_index(drop=True)
display(scores_df)


,model,MAE,RMSE,R2,MAPE
0,rf,5.116079,6.340482,0.605864,0.546797
1,cat,5.216573,6.502648,0.585445,0.539031
2,linear,5.177998,6.503359,0.585354,0.549748
3,baseline,6.500000,8.427798,0.303646,0.549318


In [15]:
joblib.dump(rf_model, f"rf_total_model_{CITY_ID}.joblib")

['rf_total_model_34.joblib']